# Dashboard BI DPPP Pariaman

Notebook ini mencakup seluruh proses pengolahan data dari csv mentah sampai masuk ke Data Mart.

1. **Extract**: membaca data mentah `data_core.csv`
2. **Transform (Cleansing)**: perbaikan typo kolom, standardisasi teks, cek anomali
3. **Transform (Enrichment)**: menambahkan kolom `Tanggal_Uji` (2023–2026) dan `Kecamatan` (4 kecamatan Kota Pariaman)
4. **Load (Star Schema)**: memecah data menjadi 5 tabel dimensi + 1 tabel fakta
5. **Load ke PostgreSQL**: membuat *datamart* (skema bintang) di PostgreSQL lengkap dengan PK/FK
6. **Verifikasi**: query contoh untuk memastikan datamart siap dikonsumsi

Anggota Kelompok:
1. Siti Aliani Husnah.F (2311522006)
2. Muhammad Abrar Rayva (2311522012)

## Setup & Konfigurasi

Instalasi library berikut:

```bash
pip install pandas numpy sqlalchemy psycopg2-binary
```

Set kredensial koneksi PostgreSQL pada `DB_CONFIG`


In [1]:
!pip install pandas numpy sqlalchemy psycopg2-binary



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Seed agar hasil reproducible
np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Konfigurasi koneksi PostgreSQL
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "data_mart_dppp_pariaman",
    "user": "postgres",
    "password": "password",
}

SCHEMA_NAME = "precision_farming"               # nama schema datamart di PostgreSQL
RAW_PATH = "../raw-dataset-csv/data_core.csv"   # path file csv mentah
OUTPUT_DIR = "../star-schema-csv"               # folder untuk menyimpan hasil CSV star schema
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Extract

Membaca data mentah `data_core.csv`.


In [3]:
df = pd.read_csv(RAW_PATH)

print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"Kolom        : {list(df.columns)}")

df.head()


Jumlah baris : 8000
Jumlah kolom : 9
Kolom        : ['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer Name']


,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,26.0,52.0,38.0,Sandy,Maize,37,0,0,Urea
1,29.0,52.0,45.0,Loamy,Sugarcane,12,0,36,DAP
2,34.0,65.0,62.0,Black,Cotton,7,9,30,14-35-14
3,32.0,62.0,34.0,Red,Tobacco,22,0,20,28-28
4,28.0,54.0,46.0,Clayey,Paddy,35,0,0,Urea


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Temparature      8000 non-null   float64
 1   Humidity         8000 non-null   float64
 2   Moisture         8000 non-null   float64
 3   Soil Type        8000 non-null   object 
 4   Crop Type        8000 non-null   object 
 5   Nitrogen         8000 non-null   int64  
 6   Potassium        8000 non-null   int64  
 7   Phosphorous      8000 non-null   int64  
 8   Fertilizer Name  8000 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 562.6+ KB


## Transform: Cleansing

- Perbaiki typo nama kolom: `Temparature` → `Temperature`, `Phosphorous` → `Phosphorus`
- Standardisasi teks kategori: *trim* spasi + *Title Case* untuk `Soil Type` & `Crop Type`
- `Fertilizer Name` di-*trim* + *uppercase* (agar kode produk seperti `DAP`, `14-35-14` tidak rusak oleh Title Case)
- Cek nilai kosong, duplikat, dan nilai numerik anomali (negatif)


In [5]:
# Perbaiki typo nama kolom
df = df.rename(columns={
    "Temparature": "Temperature",
    "Phosphorous": "Phosphorus",
})

print("Kolom setelah rename:", list(df.columns))


Kolom setelah rename: ['Temperature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorus', 'Fertilizer Name']


In [6]:
# Standardisasi teks kategori
title_case_cols = ["Soil Type", "Crop Type"]
for col in title_case_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Fertilizer Name: trim + uppercase (jaga format kode produk seperti DAP, 14-35-14)
df["Fertilizer Name"] = df["Fertilizer Name"].astype(str).str.strip().str.upper()

for col in ["Soil Type", "Crop Type", "Fertilizer Name"]:
    print(f"{col}: {sorted(df[col].unique())}")


Soil Type: ['Black', 'Clayey', 'Loamy', 'Red', 'Sandy']
Crop Type: ['Barley', 'Cotton', 'Ground Nuts', 'Maize', 'Millets', 'Oil Seeds', 'Paddy', 'Pulses', 'Sugarcane', 'Tobacco', 'Wheat']
Fertilizer Name: ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'UREA']


In [7]:
# Cek nilai kosong
print("Jumlah nilai kosong per kolom:")
print(df.isna().sum())

# Cek & hapus duplikat sempurna
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nBaris duplikat dihapus: {before - len(df)}")

# Cek nilai numerik negatif (anomali)
numeric_cols = ["Temperature", "Humidity", "Moisture", "Nitrogen", "Potassium", "Phosphorus"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        df.loc[df[col] < 0, col] = np.nan
        print(f"Nilai negatif pada '{col}' diset menjadi NaN: {neg_count} baris")

df = df.dropna(subset=numeric_cols).reset_index(drop=True)
print(f"\nData bersih final: {df.shape[0]} baris")


Jumlah nilai kosong per kolom:
Temperature        0
Humidity           0
Moisture           0
Soil Type          0
Crop Type          0
Nitrogen           0
Potassium          0
Phosphorus         0
Fertilizer Name    0
dtype: int64

Baris duplikat dihapus: 0

Data bersih final: 8000 baris


In [8]:
df.describe()


,Temperature,Humidity,Moisture,Nitrogen,Potassium,Phosphorus
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,30.338895,59.210731,43.580862,18.429125,3.916375,18.512500
std,4.478262,8.177366,12.596156,11.852406,5.494807,13.244113
min,20.000000,40.020000,20.000000,0.000000,0.000000,0.000000
25%,27.050000,53.277500,33.967500,9.000000,0.000000,8.000000
50%,30.240000,59.110000,42.250000,14.000000,1.000000,18.000000
75%,33.460000,65.082500,52.950000,26.000000,5.000000,30.000000
max,40.000000,80.000000,70.000000,46.000000,23.000000,46.000000


## Transform: Data Enrichment

Menambahkan dua kolom buatan secara logis dan terdistribusi:

- **`Tanggal_Uji`** — tanggal pengujian acak dalam rentang **2023-01-01 s.d. 2026-06-14**
- **`Kecamatan`** — salah satu dari 4 kecamatan di Kota Pariaman: *Pariaman Utara, Pariaman Tengah, Pariaman Selatan, Pariaman Timur*


In [9]:
# Injeksi kolom Tanggal_Uji
TANGGAL_AWAL = datetime(2023, 1, 1)
TANGGAL_AKHIR = datetime(2026, 6, 14)
total_hari = (TANGGAL_AKHIR - TANGGAL_AWAL).days

random_offsets = np.random.randint(0, total_hari + 1, size=len(df))
df["Tanggal_Uji"] = [TANGGAL_AWAL + timedelta(days=int(d)) for d in random_offsets]
df["Tanggal_Uji"] = pd.to_datetime(df["Tanggal_Uji"]).dt.date

# Injeksi kolom Kecamatan
KECAMATAN_LIST = [
    "Pariaman Utara",
    "Pariaman Tengah",
    "Pariaman Selatan",
    "Pariaman Timur",
]
df["Kecamatan"] = np.random.choice(KECAMATAN_LIST, size=len(df))

df[["Tanggal_Uji", "Kecamatan", "Soil Type", "Crop Type", "Fertilizer Name"]].head()


,Tanggal_Uji,Kecamatan,Soil Type,Crop Type,Fertilizer Name
0,2026-01-31,Pariaman Timur,Sandy,Maize,UREA
1,2025-05-10,Pariaman Selatan,Loamy,Sugarcane,DAP
2,2026-02-04,Pariaman Selatan,Black,Cotton,14-35-14
3,2025-12-31,Pariaman Tengah,Red,Tobacco,28-28
4,2025-11-10,Pariaman Timur,Clayey,Paddy,UREA


In [10]:
print("Distribusi data per Kecamatan:")
print(df["Kecamatan"].value_counts())

print("\nRentang Tanggal_Uji:", df["Tanggal_Uji"].min(), "s.d.", df["Tanggal_Uji"].max())


Distribusi data per Kecamatan:
Kecamatan
Pariaman Utara      2065
Pariaman Timur      2020
Pariaman Tengah     1961
Pariaman Selatan    1954
Name: count, dtype: int64

Rentang Tanggal_Uji: 2023-01-01 s.d. 2026-06-14


## Load: Pemecahan Star Schema

Dataframe utama dipecah menjadi **5 tabel dimensi** (dengan *surrogate key* `id_*`) dan **1 tabel fakta** berisi *foreign key* + *measures* numerik.

| Tabel | Tipe | Keterangan |
|---|---|---|
| `Dim_Waktu` | Dimensi | id_waktu, Tanggal_Uji, Tahun, Bulan, Nama_Bulan, Kuartal, Nama_Kuartal |
| `Dim_Kecamatan` | Dimensi | id_kecamatan, Nama_Kecamatan |
| `Dim_Tanah` | Dimensi | id_tanah, Jenis_Tanah |
| `Dim_Komoditas` | Dimensi | id_komoditas, Jenis_Tanaman |
| `Dim_Pupuk` | Dimensi | id_pupuk, Nama_Pupuk |
| `Fact_Evaluasi_Lahan` | Fakta | id_fakta + 5 FK + measures (Temperature, Humidity, Moisture, N, P, K) |


In [11]:
# DIM_WAKTU
dim_waktu = pd.DataFrame({"Tanggal_Uji": sorted(df["Tanggal_Uji"].unique())})
dim_waktu["Tanggal_Uji"] = pd.to_datetime(dim_waktu["Tanggal_Uji"])
dim_waktu["Tahun"] = dim_waktu["Tanggal_Uji"].dt.year
dim_waktu["Bulan"] = dim_waktu["Tanggal_Uji"].dt.month
dim_waktu["Nama_Bulan"] = dim_waktu["Tanggal_Uji"].dt.strftime("%B")
dim_waktu["Kuartal"] = dim_waktu["Tanggal_Uji"].dt.quarter
dim_waktu["Nama_Kuartal"] = "Q" + dim_waktu["Kuartal"].astype(str) + " " + dim_waktu["Tahun"].astype(str)
dim_waktu = dim_waktu.sort_values("Tanggal_Uji").reset_index(drop=True)
dim_waktu.insert(0, "id_waktu", range(1, len(dim_waktu) + 1))
dim_waktu["Tanggal_Uji"] = dim_waktu["Tanggal_Uji"].dt.date

dim_waktu.head()


,id_waktu,Tanggal_Uji,Tahun,Bulan,Nama_Bulan,Kuartal,Nama_Kuartal
0,1,2023-01-01,2023,1,January,1,Q1 2023
1,2,2023-01-02,2023,1,January,1,Q1 2023
2,3,2023-01-03,2023,1,January,1,Q1 2023
3,4,2023-01-04,2023,1,January,1,Q1 2023
4,5,2023-01-05,2023,1,January,1,Q1 2023


In [12]:
# DIM_KECAMATAN
dim_kecamatan = pd.DataFrame({"Nama_Kecamatan": KECAMATAN_LIST})
dim_kecamatan.insert(0, "id_kecamatan", range(1, len(dim_kecamatan) + 1))
dim_kecamatan


,id_kecamatan,Nama_Kecamatan
0,1,Pariaman Utara
1,2,Pariaman Tengah
2,3,Pariaman Selatan
3,4,Pariaman Timur


In [13]:
# DIM_TANAH
dim_tanah = pd.DataFrame({"Jenis_Tanah": sorted(df["Soil Type"].unique())})
dim_tanah.insert(0, "id_tanah", range(1, len(dim_tanah) + 1))
dim_tanah


,id_tanah,Jenis_Tanah
0,1,Black
1,2,Clayey
2,3,Loamy
3,4,Red
4,5,Sandy


In [14]:
# DIM_KOMODITAS
dim_komoditas = pd.DataFrame({"Jenis_Tanaman": sorted(df["Crop Type"].unique())})
dim_komoditas.insert(0, "id_komoditas", range(1, len(dim_komoditas) + 1))
dim_komoditas


,id_komoditas,Jenis_Tanaman
0,1,Barley
1,2,Cotton
2,3,Ground Nuts
3,4,Maize
4,5,Millets
5,6,Oil Seeds
6,7,Paddy
7,8,Pulses
8,9,Sugarcane
9,10,Tobacco


In [15]:
# DIM_PUPUK
dim_pupuk = pd.DataFrame({"Nama_Pupuk": sorted(df["Fertilizer Name"].unique())})
dim_pupuk.insert(0, "id_pupuk", range(1, len(dim_pupuk) + 1))
dim_pupuk


,id_pupuk,Nama_Pupuk
0,1,10-26-26
1,2,14-35-14
2,3,17-17-17
3,4,20-20
4,5,28-28
5,6,DAP
6,7,UREA


In [16]:
# FACT_EVALUASI_LAHAN
map_waktu = dict(zip(pd.to_datetime(dim_waktu["Tanggal_Uji"]).dt.date, dim_waktu["id_waktu"]))
map_kecamatan = dict(zip(dim_kecamatan["Nama_Kecamatan"], dim_kecamatan["id_kecamatan"]))
map_tanah = dict(zip(dim_tanah["Jenis_Tanah"], dim_tanah["id_tanah"]))
map_komoditas = dict(zip(dim_komoditas["Jenis_Tanaman"], dim_komoditas["id_komoditas"]))
map_pupuk = dict(zip(dim_pupuk["Nama_Pupuk"], dim_pupuk["id_pupuk"]))

fact_evaluasi_lahan = pd.DataFrame({
    "id_waktu": df["Tanggal_Uji"].map(map_waktu),
    "id_kecamatan": df["Kecamatan"].map(map_kecamatan),
    "id_tanah": df["Soil Type"].map(map_tanah),
    "id_komoditas": df["Crop Type"].map(map_komoditas),
    "id_pupuk": df["Fertilizer Name"].map(map_pupuk),
    "Temperature": df["Temperature"],
    "Humidity": df["Humidity"],
    "Moisture": df["Moisture"],
    "Nitrogen": df["Nitrogen"],
    "Potassium": df["Potassium"],
    "Phosphorus": df["Phosphorus"],
})
fact_evaluasi_lahan.insert(0, "id_fakta", range(1, len(fact_evaluasi_lahan) + 1))

fact_evaluasi_lahan.head()


,id_fakta,id_waktu,id_kecamatan,id_tanah,id_komoditas,id_pupuk,Temperature,Humidity,Moisture,Nitrogen,Potassium,Phosphorus
0,1,1126,4,5,4,7,26.0,52.0,38.0,37,0,0
1,2,861,3,3,9,6,29.0,52.0,45.0,12,0,36
2,3,1130,3,1,2,2,34.0,65.0,62.0,7,9,30
3,4,1095,2,4,10,5,32.0,62.0,34.0,22,0,20
4,5,1045,4,2,7,7,28.0,54.0,46.0,35,0,0


In [17]:
# Ringkasan ukuran tiap tabel star schema
star_schema_tables = {
    "Dim_Waktu": dim_waktu,
    "Dim_Kecamatan": dim_kecamatan,
    "Dim_Tanah": dim_tanah,
    "Dim_Komoditas": dim_komoditas,
    "Dim_Pupuk": dim_pupuk,
    "Fact_Evaluasi_Lahan": fact_evaluasi_lahan,
}

for name, tbl in star_schema_tables.items():
    print(f"{name:<22}: {tbl.shape}")


Dim_Waktu             : (1260, 7)
Dim_Kecamatan         : (4, 2)
Dim_Tanah             : (5, 2)
Dim_Komoditas         : (11, 2)
Dim_Pupuk             : (7, 2)
Fact_Evaluasi_Lahan   : (8000, 12)


## Simpan Backup CSV Star Schema

Sebagai cadangan / untuk diimpor manual jika diperlukan.


In [18]:
for name, tbl in star_schema_tables.items():
    tbl.to_csv(f"{OUTPUT_DIR}/{name}.csv", index=False)

print("File CSV star schema tersimpan di folder:", OUTPUT_DIR)
os.listdir(OUTPUT_DIR)


File CSV star schema tersimpan di folder: ../star-schema-csv


['Dim_Kecamatan.csv',
 'Dim_Komoditas.csv',
 'Dim_Pupuk.csv',
 'Dim_Tanah.csv',
 'Dim_Waktu.csv',
 'Fact_Evaluasi_Lahan.csv']

## Load ke PostgreSQL untuk Membangun Datamart (Star Schema)

Langkah:

1. (Opsional) Membuat database `precision_farming_db` jika belum ada
2. Membuat *engine* SQLAlchemy
3. Membuat schema `precision_farming`
4. Membuat DDL tabel dimensi & fakta lengkap dengan **Primary Key** dan **Foreign Key**
5. Memuat (`INSERT`) data dari dataframe ke masing-masing tabel — dimensi dimuat **lebih dulu** agar constraint FK pada tabel fakta valid


In [19]:
from sqlalchemy import create_engine, text

# Membuat database jika belum ada
# CREATE DATABASE tidak bisa dijalankan di dalam transaction block,
# sehingga kita koneksi ke database default 'postgres' dengan
# autocommit, lalu membuat database target jika belum tersedia.
def create_database_if_not_exists(config):
    admin_engine = create_engine(
        f"postgresql+psycopg2://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/postgres"
    )
    with admin_engine.connect() as conn:
        conn = conn.execution_options(isolation_level="AUTOCOMMIT")
        exists = conn.execute(
            text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
            {"dbname": config["dbname"]},
        ).fetchone()
        if not exists:
            conn.execute(text(f'CREATE DATABASE "{config["dbname"]}"'))
            print(f"Database '{config['dbname']}' berhasil dibuat.")
        else:
            print(f"Database '{config['dbname']}' sudah ada.")
    admin_engine.dispose()

create_database_if_not_exists(DB_CONFIG)


Database 'data_mart_dppp_pariaman' berhasil dibuat.


In [20]:
# Membuat engine SQLAlchemy ke database target
engine = create_engine(
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).scalar()
print(version)


PostgreSQL 17.0 on x86_64-windows, compiled by msvc-19.41.34120, 64-bit


In [21]:
# Membuat schema datamart
with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SCHEMA_NAME}";'))

print(f"Schema '{SCHEMA_NAME}' siap.")


Schema 'precision_farming' siap.


In [22]:
# DDL — Membuat tabel dimensi & fakta (Star Schema) dengan PK/FK
ddl_statements = f'''
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan" CASCADE;
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Dim_Waktu" CASCADE;
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Dim_Kecamatan" CASCADE;
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Dim_Tanah" CASCADE;
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Dim_Komoditas" CASCADE;
DROP TABLE IF EXISTS "{SCHEMA_NAME}"."Dim_Pupuk" CASCADE;

CREATE TABLE "{SCHEMA_NAME}"."Dim_Waktu" (
    id_waktu     INTEGER PRIMARY KEY,
    "Tanggal_Uji" DATE NOT NULL,
    "Tahun"       INTEGER NOT NULL,
    "Bulan"       INTEGER NOT NULL,
    "Nama_Bulan"  VARCHAR(20) NOT NULL,
    "Kuartal"     INTEGER NOT NULL,
    "Nama_Kuartal" VARCHAR(10) NOT NULL
);

CREATE TABLE "{SCHEMA_NAME}"."Dim_Kecamatan" (
    id_kecamatan  INTEGER PRIMARY KEY,
    "Nama_Kecamatan" VARCHAR(50) NOT NULL
);

CREATE TABLE "{SCHEMA_NAME}"."Dim_Tanah" (
    id_tanah   INTEGER PRIMARY KEY,
    "Jenis_Tanah" VARCHAR(50) NOT NULL
);

CREATE TABLE "{SCHEMA_NAME}"."Dim_Komoditas" (
    id_komoditas  INTEGER PRIMARY KEY,
    "Jenis_Tanaman" VARCHAR(50) NOT NULL
);

CREATE TABLE "{SCHEMA_NAME}"."Dim_Pupuk" (
    id_pupuk   INTEGER PRIMARY KEY,
    "Nama_Pupuk" VARCHAR(50) NOT NULL
);

CREATE TABLE "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan" (
    id_fakta      INTEGER PRIMARY KEY,
    id_waktu      INTEGER NOT NULL REFERENCES "{SCHEMA_NAME}"."Dim_Waktu"(id_waktu),
    id_kecamatan  INTEGER NOT NULL REFERENCES "{SCHEMA_NAME}"."Dim_Kecamatan"(id_kecamatan),
    id_tanah      INTEGER NOT NULL REFERENCES "{SCHEMA_NAME}"."Dim_Tanah"(id_tanah),
    id_komoditas  INTEGER NOT NULL REFERENCES "{SCHEMA_NAME}"."Dim_Komoditas"(id_komoditas),
    id_pupuk      INTEGER NOT NULL REFERENCES "{SCHEMA_NAME}"."Dim_Pupuk"(id_pupuk),
    "Temperature" NUMERIC(5,2) NOT NULL,
    "Humidity"     NUMERIC(5,2) NOT NULL,
    "Moisture"     NUMERIC(5,2) NOT NULL,
    "Nitrogen"     INTEGER NOT NULL,
    "Potassium"    INTEGER NOT NULL,
    "Phosphorus"   INTEGER NOT NULL
);

CREATE INDEX idx_fact_waktu     ON "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan"(id_waktu);
CREATE INDEX idx_fact_kecamatan ON "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan"(id_kecamatan);
CREATE INDEX idx_fact_tanah     ON "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan"(id_tanah);
CREATE INDEX idx_fact_komoditas ON "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan"(id_komoditas);
CREATE INDEX idx_fact_pupuk     ON "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan"(id_pupuk);
'''

with engine.begin() as conn:
    for statement in ddl_statements.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("DDL star schema berhasil dieksekusi.")


DDL star schema berhasil dieksekusi.


In [23]:
# Load data dari dataframe ke PostgreSQL
# Urutan WAJIB: tabel dimensi dahulu, baru tabel fakta (karena FK)
load_order = [
    ("Dim_Waktu", dim_waktu),
    ("Dim_Kecamatan", dim_kecamatan),
    ("Dim_Tanah", dim_tanah),
    ("Dim_Komoditas", dim_komoditas),
    ("Dim_Pupuk", dim_pupuk),
    ("Fact_Evaluasi_Lahan", fact_evaluasi_lahan),
]

for table_name, tbl_df in load_order:
    tbl_df.to_sql(
        name=table_name,
        con=engine,
        schema=SCHEMA_NAME,
        if_exists="append",   # tabel sudah dibuat lewat DDL di atas
        index=False,
        method="multi",
        chunksize=1000,
    )
    print(f"{table_name:<22}: {len(tbl_df)} baris dimuat ke PostgreSQL")


Dim_Waktu             : 1260 baris dimuat ke PostgreSQL
Dim_Kecamatan         : 4 baris dimuat ke PostgreSQL
Dim_Tanah             : 5 baris dimuat ke PostgreSQL
Dim_Komoditas         : 11 baris dimuat ke PostgreSQL
Dim_Pupuk             : 7 baris dimuat ke PostgreSQL
Fact_Evaluasi_Lahan   : 8000 baris dimuat ke PostgreSQL


## Verifikasi Datamart

Menjalankan beberapa query untuk memastikan data & relasi sudah benar — sekaligus mensimulasikan beberapa *measure* yang nantinya akan dipakai di dashboard Streamlit.


In [24]:
# Hitung jumlah baris setiap tabel
with engine.connect() as conn:
    for table_name, _ in load_order:
        count = conn.execute(
            text(f'SELECT COUNT(*) FROM "{SCHEMA_NAME}"."{table_name}"')
        ).scalar()
        print(f"{table_name:<22}: {count} baris")


Dim_Waktu             : 1260 baris
Dim_Kecamatan         : 4 baris
Dim_Tanah             : 5 baris
Dim_Komoditas         : 11 baris
Dim_Pupuk             : 7 baris
Fact_Evaluasi_Lahan   : 8000 baris


In [25]:
# Contoh query JOIN rata-rata N, P, K & suhu per Kecamatan
query_avg_per_kecamatan = f'''
SELECT
    k."Nama_Kecamatan",
    ROUND(AVG(f."Nitrogen")::numeric, 2)    AS avg_nitrogen,
    ROUND(AVG(f."Phosphorus")::numeric, 2)  AS avg_phosphorus,
    ROUND(AVG(f."Potassium")::numeric, 2)   AS avg_potassium,
    ROUND(AVG(f."Temperature")::numeric, 2) AS avg_temperature,
    ROUND(AVG(f."Humidity")::numeric, 2)    AS avg_humidity,
    COUNT(*) AS jumlah_sampel
FROM "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan" f
JOIN "{SCHEMA_NAME}"."Dim_Kecamatan" k ON f.id_kecamatan = k.id_kecamatan
GROUP BY k."Nama_Kecamatan"
ORDER BY k."Nama_Kecamatan";
'''

pd.read_sql(query_avg_per_kecamatan, engine)


,Nama_Kecamatan,avg_nitrogen,avg_phosphorus,avg_potassium,avg_temperature,avg_humidity,jumlah_sampel
0,Pariaman Selatan,18.40,18.82,3.95,30.44,59.37,1954
1,Pariaman Tengah,18.42,18.54,3.91,30.21,59.13,1961
2,Pariaman Timur,18.61,18.21,3.90,30.32,59.08,2020
3,Pariaman Utara,18.28,18.49,3.91,30.38,59.27,2065


In [26]:
# Contoh query frekuensi rekomendasi pupuk per kecamatan
# (akan digunakan untuk Clustered Bar Chart di dashboard)
query_pupuk_per_kecamatan = f'''
SELECT
    k."Nama_Kecamatan",
    p."Nama_Pupuk",
    COUNT(*) AS total_rekomendasi
FROM "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan" f
JOIN "{SCHEMA_NAME}"."Dim_Kecamatan" k ON f.id_kecamatan = k.id_kecamatan
JOIN "{SCHEMA_NAME}"."Dim_Pupuk" p       ON f.id_pupuk = p.id_pupuk
GROUP BY k."Nama_Kecamatan", p."Nama_Pupuk"
ORDER BY k."Nama_Kecamatan", total_rekomendasi DESC;
'''

pd.read_sql(query_pupuk_per_kecamatan, engine)


,Nama_Kecamatan,Nama_Pupuk,total_rekomendasi
0,Pariaman Selatan,UREA,312
1,Pariaman Selatan,20-20,294
2,Pariaman Selatan,14-35-14,289
3,Pariaman Selatan,28-28,278
4,Pariaman Selatan,DAP,268
5,Pariaman Selatan,17-17-17,260
6,Pariaman Selatan,10-26-26,253
7,Pariaman Tengah,14-35-14,306
8,Pariaman Tengah,28-28,292
9,Pariaman Tengah,DAP,292


In [27]:
# Contoh query tren kuartalan (akan digunakan untuk Line Chart Time-Series)
query_tren_kuartalan = f'''
SELECT
    w."Nama_Kuartal",
    w."Tahun",
    w."Kuartal",
    ROUND(AVG(f."Temperature")::numeric, 2) AS avg_temperature,
    ROUND(AVG(f."Humidity")::numeric, 2)    AS avg_humidity,
    ROUND(AVG(f."Moisture")::numeric, 2)    AS avg_moisture
FROM "{SCHEMA_NAME}"."Fact_Evaluasi_Lahan" f
JOIN "{SCHEMA_NAME}"."Dim_Waktu" w ON f.id_waktu = w.id_waktu
GROUP BY w."Nama_Kuartal", w."Tahun", w."Kuartal"
ORDER BY w."Tahun", w."Kuartal";
'''

pd.read_sql(query_tren_kuartalan, engine)


,Nama_Kuartal,Tahun,Kuartal,avg_temperature,avg_humidity,avg_moisture
0,Q1 2023,2023,1,30.51,59.02,43.06
1,Q2 2023,2023,2,30.38,59.64,43.73
2,Q3 2023,2023,3,30.31,59.19,43.50
3,Q4 2023,2023,4,30.50,59.63,43.72
4,Q1 2024,2024,1,30.43,59.21,43.41
5,Q2 2024,2024,2,30.70,59.52,43.50
6,Q3 2024,2024,3,30.10,59.20,44.14
7,Q4 2024,2024,4,30.73,58.98,44.18
8,Q1 2025,2025,1,30.25,59.44,42.95
9,Q2 2025,2025,2,30.27,59.29,44.49
